# GAS: from beamformed ultrasound frames to a collapse-pressure shift

You have beamformed ultrasound data of a gas-vesicle sample across a range of transmit pressures.
This guide takes you from those frames to a **buckling-onset pressure** per sample, and to a tested
**shift** between two samples that does not depend on matching their concentrations — the analysis
behind our [Model page](https://2026.igem.wiki/heidelberg/model#Ultrasound), running on real data so
you can see each stage before pointing it at yours.

### What your data has to provide

| | |
|---|---|
| **two ramps** | the same sample imaged over a ladder of transmit pressures **before** collapse and **after** it. The after-ramp is the background; without it there is nothing to subtract. |
| **two channels per shot** | a nonlinear one (xAM, AM, or whatever your scanner calls it) and a linear B-mode one. The linear channel locates the wells and gives the collapse check in §5. |
| **beamformed, not display** | arrays in linear units, before log compression and per-frame normalisation. Magnitudes are enough — nothing here uses the phase. |
| **axes in mm** | a lateral and a depth axis, so a region of interest can be given in millimetres. |
| **a pressure calibration** | your probe's transmit setting → pressure. Ours is a hydrophone map for a GE9LD at 5.21 MHz; yours will differ, and §3 shows where to swap it. |

Anything that meets this runs through unchanged — §1 shows how to wrap your own arrays. What you do
**not** need: our scanner, our file format, our mold, or our probe.

### To run it

Install `numpy scipy pandas matplotlib h5py` and execute the cells top to bottom. Everything else is
in this folder: the analysis package, the lab-book tables, and 21 MB of example data.

The example is `0925/Mold2` — purified *Serratia* gas vesicles **+GvpC** and **−GvpC** plus a PBS
control, cast in an agarose mold with three wells, imaged on a Verasonics Vantage with a GE9LD probe
at 5.21 MHz, over a transmit ramp of 2–50 V before and after collapsing the vesicles.

In [ ]:
import os, sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()                                  # gvv/, labbook/ and data/ sit next to this notebook
sys.path.insert(0, str(HERE))
os.environ.setdefault("GVV_DATA", str(HERE / "data"))
warnings.filterwarnings("ignore")                  # load-time notes about persistence/axes, printed per file

from gvv import claims, collapse, pressure, store
from gvv import roi as R

MOLD = "0925/Mold2"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": .25})

## 1 · The beamformed data

We keep the beamformed IQ (`BFData_raw`), not the screen image. One acquisition is two arrays of
shape (depth, lateral ray) in linear units:

* `pw` — **plane wave**, an ordinary B-mode: everything that scatters. The *linear* channel.
* `xam` — **cross-amplitude modulation**: X is transmitted, then its halves L and R, and X−L−R is
  kept. Linear scatterers cancel; what survives is *nonlinear* signal, i.e. buckling vesicles.

Everything below quantifies `np.abs(...)` of these — never the exported PNGs or CSVs, which are
log-compressed and normalised per frame. Since only the magnitude is used, the bundled data stores
`|xAM|` and `|PW|` as float32: same numbers, a fifth of the size.

In [ ]:
pre = R.load_ramp(MOLD)              # the intact ramp, sorted by transmit voltage
ac = R.load_ramp(MOLD + "_AC")       # the same wells after collapse
a = next(x for x in pre if x.voltage == 12.0)

print(f"{len(pre)} pre frames, {len(ac)} AC frames, {pre[0].voltage:g}–{pre[-1].voltage:g} V")
print(f"one frame: {a.xam.shape} (depth × ray), {a.xam.dtype}, source {a.raw_source}")
print(f"axes: x {a.x_mm[0]:.1f} … {a.x_mm[-1]:.1f} mm, z {a.z_mm[0]:.1f} … {a.z_mm[-1]:.1f} mm")
print(f"{a.voltage:g} V transmit = {pressure.kpa(a.voltage):.0f} kPa peak positive pressure")

Both channels of one shot, in dB re. each image's maximum. The shaded strip is outside the
cross-wave overlap (`xam_valid`) — no xAM contrast exists there, and those pixels never enter a ROI.

In [ ]:
def show(ax, img, a, title, dyn=40, valid=None):
    db = 20 * np.log10(np.abs(img) / np.abs(img).max() + 1e-12)
    ext = [a.x_mm[0], a.x_mm[-1], a.z_mm[-1], a.z_mm[0]]
    im = ax.imshow(db, extent=ext, vmin=-dyn, vmax=0, cmap="gray", aspect="equal")
    if valid is not None:
        ax.imshow(np.where(valid, np.nan, 1), extent=ext, cmap="autumn", alpha=.18, aspect="equal")
    ax.set(title=title, xlabel="x (mm)", ylabel="depth (mm)")
    ax.grid(False)
    return im

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), constrained_layout=True)
im = show(axes[0], a.pw, a, f"plane wave (linear) · {a.voltage:g} V")
show(axes[1], a.xam, a, f"xAM (nonlinear) · {a.voltage:g} V", valid=a.xam_valid)
fig.colorbar(im, ax=axes, label="dB re. frame max", shrink=.9)
plt.show()

### Your own frames

The loader above reads our store, but nothing downstream knows or cares where the arrays came from.
Every later step takes a plain record: two image arrays shaped (depth, lateral), the two axes in mm,
the transmit setting of that shot, and a mask of where the nonlinear channel is valid. Build that
from your own arrays and the rest of this notebook works as written.

In [ ]:
from gvv.io import Acq


def record(xam, pw, x_mm, z_mm, transmit, is_ac, valid=None):
    """Wrap your own beamformed frame so the pipeline can read it.

    xam, pw : (depth, lateral) arrays in linear units — complex IQ or magnitudes, both fine
    x_mm, z_mm : the lateral and depth axes, same lengths as the array's two sides
    transmit : the transmit setting of this shot (volts here; any monotone scale works)
    is_ac : False before collapse, True after
    valid : optional bool mask, False where the nonlinear channel has no contrast
    """
    return Acq(path="", generation="own", domain="iq", round="", mold="", sample=None,
               is_ac=is_ac, voltage=float(transmit), xam=xam, pw=pw, x_mm=x_mm, z_mm=z_mm,
               xam_valid=np.ones(xam.shape, bool) if valid is None else valid)


# demonstrated on four frames of the example, as if they had arrived as bare arrays
mine = [record(f.xam, f.pw, f.x_mm, f.z_mm, f.voltage, False) for f in pre[:4]]
mine_ac = [record(f.xam, f.pw, f.x_mm, f.z_mm, f.voltage, True) for f in ac[:4]]
own_roi = R.find_rois(mine, mine_ac)
print("wells found at x =", [f"{l:.1f}…{r:.1f} mm" for l, r in own_roi["wells"]])
print(R.ramp_curves(mine, mine_ac, own_roi, ["A", "B", "C"], stat="median",
                    quantity="intensity")[["well", "V", "xam_sub"]].head(3).to_string(index=False))

## 2 · Regions of interest

Three wells sit at x ≈ −12, 0, +12 mm, 13–17 mm deep. `find_rois` takes the depth band from the
agarose interface and the mold floor, and finds the wells as the three wide bright runs in the
lateral dB profile.

Purified vesicles at low concentration are nearly anechoic, so detection fails here and the hand-set
ROIs from the lab book are used (`labbook/roi_overrides.csv`) — a research decision, so it is stored
rather than re-derived. Well identities come from the lab book, left to right.

**For your data:** the detector expects three wells side by side. With a different geometry, pass the
ROIs yourself — `roi["wells"]` is just a list of `(x_left, x_right)` in mm and `roi["band"]` a boolean
mask over the depth axis, so you can set both by hand and skip `find_rois` entirely. Look at the
figure below before trusting anything downstream of it: a wrong ROI invalidates every later number.

In [ ]:
roi = R.find_rois(pre, ac, folder=MOLD)
labels = [w["label"] for w in store.well_labels(MOLD)]
z = pre[0].z_mm

print("ROI source:", roi["note"] or "automatic detection")
for lab, (l, r) in zip(labels, roi["wells"]):
    print(f"  {lab:>6}: x {l:+.2f} … {r:+.2f} mm")
print(f"  depth band {z[roi['band']][0]:.1f} … {z[roi['band']][-1]:.1f} mm "
      f"({roi['band'].sum()} rows), interface {roi['top']:.1f} mm, floor {roi['floor']:.1f} mm")

In [ ]:
fig, (ax, bx) = plt.subplots(2, 1, figsize=(6.4, 5), height_ratios=[2, 1], constrained_layout=True)
show(ax, roi["mpw"], pre[0], "mean plane-wave power over both ramps, with the ROIs")
for lab, (l, r) in zip(labels, roi["wells"]):
    ax.add_patch(plt.Rectangle((l, z[roi["band"]][0]), r - l, np.ptp(z[roi["band"]]),
                               ec="deepskyblue", fc="none", lw=1.4))
    ax.text((l + r) / 2, z[roi["band"]][0] - .6, lab, color="deepskyblue", ha="center", fontsize=8)
ax.axhline(roi["top"], color="orange", lw=.8)

bx.plot(pre[0].x_mm, roi["prof"], lw=1)
bx.axhline(roi["thr"], color="crimson", ls="--", lw=.8, label="detection threshold")
for l, r in roi["wells"]:
    bx.axvspan(l, r, color="deepskyblue", alpha=.18)
bx.set(xlabel="x (mm)", ylabel="dB in the band", title="lateral profile (wells shaded)")
bx.legend(fontsize=8)
plt.show()

## 3 · Summary statistics

`ramp_curves` pairs each pre frame with the AC frame at the same voltage and reduces every ROI to a
few numbers. Three choices matter:

* **median** of |IQ|², not mean: speckle is heavy-tailed, one bright fleck would move a mean.
* **background subtracted in intensity**, `xam_sub = median|xAM_pre|² − median|xAM_AC|²`. Intensities
  add, amplitudes do not. Not clipped at zero — below background it goes negative, and that is noise;
  clipping biased the quiet low-pressure end upward.
* **a measured error bar**: `speckle_se`, a MAD spread over the number of *independent* speckle cells,
  with pixels-per-speckle measured from the autocorrelation rather than assumed. Within-frame
  measurement noise, not a biological SEM.

Voltage becomes pressure through a hydrophone calibration (`pressure.kpa`): peak positive pressure in
water at the well depth, not derated, so an upper bound on what the vesicles feel.

**For your data:** that map is specific to our probe and sequence. Replace `pressure.kpa` with your
own transmit → pressure function, or keep working in the transmit unit — every fit below takes an
x axis, not a voltage, and `kpa=False` fits on the raw setting. Only the absolute pressures change;
a ratio or a within-mold shift does not.

In [ ]:
d = R.ramp_curves(pre, ac, roi, labels, dataset=MOLD, stat="median", quantity="intensity")
d = d.assign(xam_sub=d.xam_sub - min(d.xam_sub.min(), 0))   # lift the whole mold so the log axis works
d["kPa"] = pressure.kpa(d.V)
d["se"] = R.curve_se(d, "xam_sub")

print(f"px per speckle cell, measured: {d.px_per_speckle.min():.1f}–{d.px_per_speckle.max():.1f}")
d.loc[d.well == "+GvpC", ["well", "V", "kPa", "xam_stat", "xam_stat_ac", "xam_sub", "se"]].head(4)

In [ ]:
fig, (ax, bx) = plt.subplots(1, 2, figsize=(9.4, 3.4), constrained_layout=True)
for lab, g in d.groupby("well", sort=False):
    ax.plot(g.kPa, g.xam_stat, "-o", ms=3, lw=1, label=f"{lab} pre")
    ax.plot(g.kPa, g.xam_stat_ac, "--", lw=.9, color=ax.lines[-1].get_color(), label=f"{lab} after collapse")
    bx.errorbar(g.kPa, g.xam_sub, g.se, fmt="o", ms=3, lw=.8, capsize=1.5, label=lab)
ax.set(xlabel=pressure.LABEL, ylabel="median |xAM|² in ROI (a.u.)", title="raw ramps, pre and after collapse")
bx.set(xlabel=pressure.LABEL, ylabel="pre − AC (a.u.)", yscale="log", title="the signal we fit")
ax.legend(fontsize=7, ncols=2); bx.legend(fontsize=8)
plt.show()

In [ ]:
# PBS is the no-vesicle control: never fitted, only the two GV wells go on
gv = d[~d.well.str.lower().str.contains("pbs")]
se = R.curve_se(gv, "xam_sub")

## 4 · The GAS model

Read naively, an onset is the pressure where signal clears background — which depends on how many
vesicles are in the well. GAS factorises the curve instead. Signal ∝ (how many are active) × (how
much each gives), and vesicles only emit nonlinear signal once the shell **buckles**, and nothing at
all once it **collapses**:

$$\mathrm{xAM}(P) \;=\; b_0 \;+\; A \cdot \underbrace{\mathrm{rise}(P;\,P_{50},\,w)}_{\text{buckling activation}} \cdot \underbrace{\bigl(1 - B\,\mathrm{rise}(P;\,P_{50}+\delta,\,w_d)\bigr)}_{\text{nuisance}}$$

Concentration sits in $A$ and drops out. The **onset** belongs to the activation term alone,
$P_\mathrm{onset} = P_{50} - 2w$ (the tangent at the steepest point meets the baseline); the 10–90 %
**width** is $2\ln 9 \cdot w$.

The second factor is a **nuisance** term, not a collapse curve: it absorbs vesicle loss during the
ramp, the growing response per vesicle, the voltage-to-pressure law, xAM saturation, the mold, the
probe. Its shape was fixed in advance — falling sigmoid, starting at 1, centred after $P_{50}$ so it
cannot eat the rise — and its parameters are never interpreted.

Seven parameters on ~40 points, so the fit rules were fixed before any claim (`fit_peak`,
`fit_check`): window from the first point to peak + 1.5 × (peak − first point), ≥ 8 points or no fit,
deterministic starts, weights from the speckle SE, errors scaled by √χ²_red when scatter exceeds it,
and any fit showing a known failure signature is dropped from every test.

In [ ]:
def gas(P, b0, A, p50, w, B, dl, wd):
    """The published model, for reference — the fits below call gvv.collapse.peak_model."""
    return b0 + A * collapse.rise(P, p50, w) * (1 - B * collapse.rise(P, p50 + dl, wd))

P = np.linspace(0, 3000, 400)
fig, ax = plt.subplots(figsize=(5.2, 3), constrained_layout=True)
ax.plot(P, collapse.rise(P, 1400, 120), label="activation: buckling")
ax.plot(P, 1 - .85 * collapse.rise(P, 1800, 250), label="nuisance: survival etc.")
ax.plot(P, gas(P, 0, 1, 1400, 120, .85, 400, 250), "k", lw=1.8, label="product = GAS")
ax.axvline(1400 - 2 * 120, color="crimson", ls=":", lw=1, label="onset = $P_{50}-2w$")
ax.set(xlabel="pressure (kPa)", ylabel="relative signal", title="the two factors and their product")
ax.legend(fontsize=8)
plt.show()

Fitted per well in pressure, with the window rule applied on the transmit voltages (our calibration
flattens, so a ×2.5 window in kPa would run off the end of the ramp). **For your data:** the window
factor is a researcher degree of freedom — fix it before you look at any comparison, and say which
value you used.

In [ ]:
ends = {k.split("|", 1)[1]: v for k, v in claims.load_rise().items()
        if k.split("|", 1)[0] == MOLD and v > 0}          # any hand-set window ends for this mold
fits = collapse.rise_fits(gv, "xam_sub", se, ends)
fits[["well", "onset", "onset_se", "v50", "v50_se", "width", "width_se", "n", "chi2_red", "check"]].round(1)

`−GvpC` fails its fit check: the nuisance factor is pushed against $P_{50}$, so activation and
nuisance are not separated. Per the rules it is not quoted alone — which is what the **parallel fit**
is for. Both wells keep their own baseline, amplitude and nuisance but **share one width**, so the
only thing separating them is a shift along the pressure axis.

In [ ]:
pf = collapse.parallel_fit(gv, "xam_sub", se, fits)
r = pf.iloc[0]
shift, shift_se = r["shift"], r["shift_se"]   # r.shift would be the DataFrame method
print(f"shift ({r.well_a} − {r.well_b}): {shift:+.0f} ± {shift_se:.0f} kPa  "
      f"[{r.shift_lo:.0f}, {r.shift_hi:.0f}], p = {r.p:.1e}")
print(f"P50 ratio {r.well_a}/{r.well_b}: {r.ratio:.3f} ± {r.ratio_se:.3f}, p = {r.p_ratio:.1e}")
print(f"shared width {collapse.WIDTH * r.w:.0f} kPa · χ²_red {r.chi2_red:.1f} on {r.dof:.0f} dof")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.6), constrained_layout=True)
for (lab, g), pars in zip(gv.groupby("well", sort=False), (r.params_a, r.params_b)):
    f = fits[fits.well == lab].iloc[0]
    m = g.kPa <= f.v_max
    ax.errorbar(g.kPa, g.xam_sub, R.curve_se(g, "xam_sub"), fmt="o", ms=3.5, lw=.8, capsize=1.5, label=lab)
    col = ax.lines[-1].get_color()
    x = np.linspace(g.kPa[m].min(), f.v_max, 300)
    ax.plot(x, collapse.peak_model(x, *pars), color=col, lw=1.4)
    ax.axvline(pars[2] - 2 * pars[3], color=col, ls=":", lw=1)   # onset = P50 − 2w of the parallel fit
ax.set(xlabel=pressure.LABEL, ylabel="xAM pre − AC (a.u.)", yscale="log",
       title="parallel fit · dotted lines: onsets")
ax.legend(fontsize=8)
plt.show()

### Is the shift real, or an artefact of the model?

**Hydrostatic** collapse contains no frequency, so the ratio of hydrostatic midpoints is a
frequency-free ground truth for which sample is more stable, and a ratio drops any unknown
hydrostatic-to-acoustic conversion factor.

The partner curve is the *acoustic collapse* curve — a Boltzmann fit to the **linear** plane-wave
signal (`collapse.pw_excess`: how much collapsible scatterer is left), kept separate from the
nonlinear signal GAS fits so the check is not circular. `gas.fig1_data` builds both.

In [ ]:
from gvv import gas

hyd_pts, aco_pts, step1 = gas.fig1_data()        # hydrostatic points, acoustic points, fitted table
step1[["modality", "condition", "p50", "p50_se", "p50_native", "ratio", "ratio_se", "n", "r2_adj"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True, constrained_layout=True)
for ax, (mod, pts) in zip(axes, (("hydrostatic", hyd_pts), ("acoustic", aco_pts))):
    for cond, g in pts.groupby("condition"):
        ax.errorbar(g.kpa, g.y, g.err, fmt="o", ms=3, lw=.6, capsize=1.5, alpha=.7, label=cond)
        f = step1[(step1.modality == mod) & (step1.condition == cond)].iloc[0]
        x = np.linspace(max(pts.kpa.min(), 1), pts.kpa.max(), 300)
        ax.plot(x, gas.boltzmann(x, f.a, f.p50, f.s, f.c), color=ax.lines[-1].get_color(), lw=1.3)
    ax.set(xlabel="pressure (kPa)", title=f"{mod} collapse")
axes[0].set_ylabel("portion of intact GVs")
axes[0].legend(fontsize=8)
plt.show()

The three ratios, against the Model page.

In [ ]:
step = step1.drop_duplicates("modality").set_index("modality")
hydro, acou = step.loc["hydrostatic"], step.loc["acoustic"]
print(f"                                          here            Model page")
print(f"Step 1  hydrostatic collapse ratio     : {hydro.ratio:.3f} ± {hydro.ratio_se:.3f}   1.47 ± 0.04")
print(f"Step 1  acoustic collapse ratio        : {acou.ratio:.3f} ± {acou.ratio_se:.3f}   1.52 ± 0.05   (linear signal)")
print(f"Step 2  GAS activation midpoint ratio  : {r.ratio:.3f} ± {r.ratio_se:.3f}   1.45 ± 0.10   (nonlinear signal)")
print(f"\nactivation vs hydrostatic: "
      f"{abs(r.ratio - hydro.ratio) / np.hypot(r.ratio_se, hydro.ratio_se):.1f} SE from the hydrostatic ground truth")

## 5 · Does the fit recover what it is told?

Two synthetic curves from `peak_model` with a known shift, sampled at this mold's real voltages with
noise at the measured scale, through the same pipeline.

In [ ]:
rng = np.random.default_rng(0)
V = np.sort(gv[gv.well == "+GvpC"].V.unique())
kpa = pressure.kpa(V)
TRUE = dict(b0=3e4, A=4e6, p50=1200.0, w=120.0, B=.85, dl=400.0, wd=250.0)
SHIFT = 450.0                                     # kPa, the quantity we want back

truth, rows = {}, []
for lab, shift in (("sim +", SHIFT), ("sim −", 0.0)):
    p = (TRUE["b0"], TRUE["A"], TRUE["p50"] + shift, TRUE["w"], TRUE["B"], TRUE["dl"], TRUE["wd"])
    y = collapse.peak_model(kpa, *p)
    noise = .04 * y.max()
    truth[lab] = dict(onset=p[2] - 2 * p[3], v50=p[2], width=collapse.WIDTH * p[3])
    rows += [dict(dataset="sim", well=lab, V=v, xam_sub=yi + rng.normal(0, noise), se=noise)
             for v, yi in zip(V, y)]
sim = pd.DataFrame(rows)

sfits = collapse.rise_fits(sim, "xam_sub", sim.se, {})
spf = collapse.parallel_fit(sim, "xam_sub", sim.se, sfits).iloc[0]

out = sfits.set_index("well")[["onset", "onset_se", "v50", "v50_se", "width", "width_se"]]
out.insert(0, "true onset", [truth[w]["onset"] for w in out.index])
out.insert(4, "true v50", [truth[w]["v50"] for w in out.index])
print(out.round(1).to_string(), "\n")
got, got_se = spf["shift"], spf["shift_se"]
print(f"recovered shift {got:+.0f} ± {got_se:.0f} kPa   (true {SHIFT:+.0f})")

assert abs(got - SHIFT) < 3 * got_se, "parallel fit does not recover the injected shift"
for w, t in truth.items():
    f = sfits.set_index("well").loc[w]
    assert abs(f.v50 - t["v50"]) < 3 * max(f.v50_se, 1), f"{w}: P50 off"
print("checks passed")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2), constrained_layout=True)
for (lab, g), col in zip(sim.groupby("well", sort=False), ("C0", "C1")):
    f = sfits.set_index("well").loc[lab]
    ax.errorbar(pressure.kpa(g.V), g.xam_sub, g.se, fmt="o", ms=3, lw=.8, color=col, label=lab)
    x = np.linspace(f.v_min, f.v_max, 300)
    ax.plot(x, collapse.peak_model(x, f.b0, f.a, f.v50, f.w, f.b, f.dl, f.wd), color=col, lw=1.3)
ax.set(xlabel=pressure.LABEL, ylabel="simulated xAM (a.u.)", title="synthetic curves and their fits")
ax.legend(fontsize=8)
plt.show()

## Checklist for your own run

1. **Wrap your frames** with `record(...)` from §1, one per shot, pre and after collapse.
2. **Set the ROIs** — `find_rois` if your mold is three wells in a row, by hand otherwise. Check the
   figure.
3. **Swap the pressure map**, or work in your transmit unit with `kpa=False`.
4. **Fix the fit window** before comparing anything, and declare it.
5. **Read the fit-check column** before quoting a fit; a flagged well is not reported alone.
6. **Compare within one mold only** — $P_{50}$ depends on the nuisance shape, which is not shared
   across phantoms.
7. **Sanity-check the estimator** on synthetic curves at your own noise level (§5).

## Limits

* **One mold.** A second purified mold (`0922/Mold1`) points the other way and the bacterial mold
  shows nothing, so the page makes no GvpC claim on this data alone. `scripts/40_parallel_claims.py`
  in the working repo runs every mold of a claim.
* **The nuisance term is not a collapse curve** — not interpreted, and the B-mode survival proxy
  disagrees with it. $P_{50}$ depends on its shape, so only *within-mold* comparisons are made.
* **Pressures are upper bounds**: water calibration, no derating through agarose, ~9 % hydrophone
  uncertainty not in the error bars, one trace per voltage.
* **Error bars are speckle** — within-frame noise. One mold has no biological replication.